# Python Mastery — Module 6
# Supervised Machine Learning Algorithms: Beginner to Advanced

This module explains the major supervised-learning algorithms, the intuition behind them, their assumptions, strengths, weaknesses, important hyperparameters, and how to compare them fairly.

## Learning outcomes
- Understand linear and logistic regression beyond simply calling `.fit()`.
- Explain K-nearest neighbors, decision trees, random forests, bagging, boosting, gradient boosting, SVMs, and naive Bayes.
- Know when scaling, encoding, or probability calibration matters.
- Compare models with cross-validation and appropriate metrics.
- Interpret model complexity, regularization, feature importance, and error patterns.
- Build a reproducible model-selection workflow.

**Examples:** synthetic datasets for classification and regression. Results demonstrate methods, not guarantees for real-world data.

## 1. A quick map of supervised algorithms

| Algorithm | Main idea | Often useful when | Common caution |
|---|---|---|---|
| Linear regression | Fit a linear relationship to a numeric target | Baselines, interpretable numeric prediction | Outliers, nonlinearity, correlated predictors |
| Logistic regression | Model class probability through a logistic link | Strong tabular baseline, interpretable classification | Linear decision boundary unless features are engineered |
| K-nearest neighbors (KNN) | Predict from nearby training examples | Small/medium data with meaningful distances | Scaling, high dimensions, prediction cost |
| Decision tree | Split features into regions using rules | Nonlinear patterns and readable rules | Deep trees overfit |
| Random forest | Average many randomized trees | Strong general-purpose tabular baseline | Less transparent than one tree; larger models |
| Bagging | Train models on bootstrap samples and aggregate | Reduce variance of unstable learners | Does not automatically reduce bias |
| Boosting | Add learners sequentially to correct errors | Strong tabular prediction | Sensitive tuning; can overfit/noisy labels |
| Gradient boosting | Optimize a loss by adding small learners stage by stage | Structured/tabular data | Requires tuning and careful validation |
| SVM | Find a maximum-margin boundary; kernels allow nonlinear boundaries | Medium-sized data with clear margins | Scaling and parameter tuning; probability estimates may need calibration |
| Naive Bayes | Apply Bayes' theorem with conditional-independence assumption | Text, high-dimensional sparse features, fast baselines | Independence assumption can be unrealistic |

No algorithm is universally best. Data shape, sample size, feature types, target definition, latency, interpretability, and error costs all matter.

## 2. Set up the data

We will create two datasets:
- a regression dataset with a continuous target;
- a binary classification dataset with an imbalanced positive class.

Use consistent random seeds to make demonstrations repeatable. Different library versions may produce small differences.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB

RANDOM_STATE = 42
Xr, yr = make_regression(
    n_samples=1800, n_features=10, n_informative=7,
    noise=22, random_state=RANDOM_STATE
)
Xr = pd.DataFrame(Xr, columns=[f"x{i}" for i in range(Xr.shape[1])])

Xc, yc = make_classification(
    n_samples=2400, n_features=14, n_informative=7,
    n_redundant=3, weights=[0.87, 0.13], class_sep=0.9,
    random_state=RANDOM_STATE
)
Xc = pd.DataFrame(Xc, columns=[f"feature_{i}" for i in range(Xc.shape[1])])
print("Regression:", Xr.shape, "Classification:", Xc.shape)
print("Positive class prevalence:", round(yc.mean(), 3))

## 3. Linear regression

Linear regression predicts a numeric outcome using a weighted combination of input features.

For one feature, the model is a line. With multiple features, it fits a hyperplane. The usual least-squares objective minimizes the sum of squared residuals.

**Terms**
- **Intercept:** predicted value when all encoded predictors equal zero.
- **Coefficient:** expected change in the model prediction for a one-unit feature increase, holding other included predictors constant.
- **Prediction:** model output.
- **Residual:** observed target minus predicted target.

Coefficients are associations under the fitted model, not automatically causal effects. Their interpretation depends on feature units, coding, confounding, and model assumptions.

### Linear regression assumptions and diagnostics

Common classical assumptions concern linearity of the conditional mean, independent errors, constant error variance, and—when using certain small-sample inference procedures—error distribution assumptions. Predictive use does not require every assumption to hold perfectly, but violations can harm reliability and inference.

Check:
- residuals versus predictions for patterns or changing variance;
- residual distribution for unusual tails;
- leverage and influential observations;
- correlated predictors and coefficient instability;
- train/test error for generalization.

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=RANDOM_STATE
)
linear = LinearRegression()
linear.fit(Xr_train, yr_train)
pred_linear = linear.predict(Xr_test)
residuals = yr_test - pred_linear

print("MAE:", round(mean_absolute_error(yr_test, pred_linear), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(yr_test, pred_linear)), 2))
print("R²:", round(r2_score(yr_test, pred_linear), 3))
print("Intercept:", round(linear.intercept_, 2))
pd.Series(linear.coef_, index=Xr.columns, name="coefficient").sort_values(key=abs, ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(pred_linear, residuals, alpha=0.55)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted target")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs predictions")
axes[1].scatter(yr_test, pred_linear, alpha=0.55)
axes[1].set_xlabel("Actual target")
axes[1].set_ylabel("Predicted target")
axes[1].set_title("Actual vs predicted")
plt.tight_layout()
plt.show()

## 4. Regularized linear models: Ridge and Lasso

When features are numerous or correlated, ordinary least squares can produce unstable coefficients.

- **Ridge (L2):** penalizes the sum of squared coefficients. It shrinks coefficients toward zero but typically does not set them exactly to zero.
- **Lasso (L1):** penalizes the sum of absolute coefficients and can set some coefficients to zero.
- **Elastic Net:** combines L1 and L2 penalties (available in scikit-learn as `ElasticNet`).

The regularization strength controls the trade-off between fitting the training data and keeping the model simpler. Scaling is important because penalties depend on coefficient sizes.

In [ ]:
from sklearn.linear_model import ElasticNet

regressors = {
    "Linear": Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=10.0))]),
    "Lasso": Pipeline([("scale", StandardScaler()), ("model", Lasso(alpha=2.0, max_iter=10000))]),
    "Elastic Net": Pipeline([("scale", StandardScaler()), ("model", ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))]),
}
rows = []
for name, estimator in regressors.items():
    estimator.fit(Xr_train, yr_train)
    p = estimator.predict(Xr_test)
    rows.append({"model": name, "MAE": mean_absolute_error(yr_test, p),
                 "RMSE": np.sqrt(mean_squared_error(yr_test, p)), "R2": r2_score(yr_test, p)})
pd.DataFrame(rows).set_index("model").round(3)

## 5. Logistic regression

Despite its name, logistic regression is commonly used for **classification**. It models a linear score and passes it through a logistic function to produce a probability between 0 and 1.

The default binary decision rule is commonly 0.5, but the threshold should be selected for the application. `C` is the inverse regularization strength in scikit-learn: smaller `C` means stronger regularization.

Strengths:
- efficient and strong baseline;
- coefficients can be interpreted on the log-odds scale;
- often provides useful probabilities, although calibration should be checked.

Limitations:
- the default boundary is linear in the transformed features;
- interactions and nonlinear effects may require feature engineering;
- correlated features can make coefficients less stable.

### Log-odds interpretation

For a one-unit increase in a numeric feature, the coefficient changes the model's log odds by that coefficient, holding other model features fixed. Exponentiating a coefficient gives an odds ratio for a one-unit increase. An odds ratio is not the same as a probability multiplier.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, stratify=yc, random_state=RANDOM_STATE
)
logistic = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
])
logistic.fit(Xc_train, yc_train)
logit_pred = logistic.predict(Xc_test)
logit_proba = logistic.predict_proba(Xc_test)[:, 1]
print("Accuracy:", round(accuracy_score(yc_test, logit_pred), 3))
print("Precision:", round(precision_score(yc_test, logit_pred, zero_division=0), 3))
print("Recall:", round(recall_score(yc_test, logit_pred, zero_division=0), 3))
print("F1:", round(f1_score(yc_test, logit_pred, zero_division=0), 3))
print("ROC-AUC:", round(roc_auc_score(yc_test, logit_proba), 3))
print("Average precision:", round(average_precision_score(yc_test, logit_proba), 3))

## 6. K-nearest neighbors (KNN)

KNN predicts using the labels or values of nearby training examples:
1. Choose a distance metric.
2. Find the k closest training points.
3. Vote (classification) or average (regression) their outcomes.

- Small `k`: flexible boundary, sensitive to noise, can overfit.
- Large `k`: smoother boundary, may underfit.
- Feature scaling is important because distance is dominated by large-scale features otherwise.
- In high-dimensional spaces, distances become less informative (“curse of dimensionality”).
- Prediction can be expensive because many training points may need to be compared.

Try moving the query point and changing k to see how neighbor voting changes.

### Interactive KNN concept

In [ ]:
knn_results = []
for k in [1, 3, 5, 11, 25]:
    knn = Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k, weights="uniform")),
    ])
    knn.fit(Xc_train, yc_train)
    p = knn.predict(Xc_test)
    s = knn.predict_proba(Xc_test)[:, 1]
    knn_results.append({"k": k, "accuracy": accuracy_score(yc_test, p),
                        "recall": recall_score(yc_test, p, zero_division=0),
                        "F1": f1_score(yc_test, p, zero_division=0),
                        "average_precision": average_precision_score(yc_test, s)})
pd.DataFrame(knn_results).set_index("k").round(3)

## 7. Decision trees

A decision tree learns a sequence of feature-based questions. At each split, it chooses a rule that improves a criterion such as Gini impurity or entropy for classification.

Key controls:
- `max_depth`: limits how many levels the tree can grow.
- `min_samples_split`: minimum rows needed to split a node.
- `min_samples_leaf`: minimum rows allowed in a leaf.
- `criterion`: measure used to evaluate split quality.

Trees capture nonlinear relationships and interactions without requiring feature scaling. Deep trees can memorize training examples and are often high-variance models.

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=12,
    class_weight="balanced", random_state=RANDOM_STATE
)
tree.fit(Xc_train, yc_train)
tree_pred = tree.predict(Xc_test)
print("Tree F1:", round(f1_score(yc_test, tree_pred, zero_division=0), 3))
plt.figure(figsize=(16, 7))
plot_tree(tree, feature_names=Xc.columns, class_names=["Negative", "Positive"],
          filled=False, rounded=True, max_depth=3, fontsize=8)
plt.title("Decision tree (visualized up to depth 3)")
plt.show()

## 8. Bagging and random forests

### Bagging
Bootstrap aggregating trains many models on bootstrap samples of the training data and combines their predictions. It mainly reduces variance when the base learner is unstable.

### Random forest
A random forest adds random feature subsets at each split, making individual trees less correlated. Aggregating many diverse trees often gives a strong tabular baseline.

Important parameters:
- `n_estimators`: number of trees.
- `max_depth`, `min_samples_leaf`: tree complexity controls.
- `max_features`: features considered at each split.
- `class_weight`: adjusts the training penalty for class imbalance.

Tree ensembles generally do not require standardization. They can model nonlinearities but are less straightforward to explain than a single small tree.

In [ ]:
forest_models = {
    "Single tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, class_weight="balanced", random_state=RANDOM_STATE),
    "Bagging trees": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None, min_samples_leaf=5, random_state=RANDOM_STATE),
        n_estimators=100, max_samples=0.8, bootstrap=True, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=250, min_samples_leaf=4, max_features="sqrt",
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Extra trees": ExtraTreesClassifier(
        n_estimators=250, min_samples_leaf=4, max_features="sqrt",
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
}
rows = []
for name, model in forest_models.items():
    model.fit(Xc_train, yc_train)
    p = model.predict(Xc_test)
    s = model.predict_proba(Xc_test)[:, 1]
    rows.append({"model": name, "precision": precision_score(yc_test, p, zero_division=0),
                 "recall": recall_score(yc_test, p, zero_division=0),
                 "F1": f1_score(yc_test, p, zero_division=0),
                 "ROC-AUC": roc_auc_score(yc_test, s),
                 "average_precision": average_precision_score(yc_test, s)})
pd.DataFrame(rows).set_index("model").round(3)

## 9. Boosting: AdaBoost, gradient boosting, and histogram gradient boosting

Boosting builds learners sequentially so later learners improve the current ensemble. Unlike bagging, which trains bootstrap learners largely independently, boosting focuses on residual errors or the loss gradient.

- **AdaBoost:** increases emphasis on examples that previous weak learners handled poorly.
- **Gradient boosting:** adds small trees to reduce a chosen loss function.
- **Histogram gradient boosting:** bins numeric features to speed up gradient boosting on larger tabular datasets.

Important controls include number of boosting stages, learning rate, tree depth/leaf size, and subsampling. A smaller learning rate often needs more estimators. Boosting can be powerful but should be tuned with cross-validation and checked for overfitting.

In [ ]:
boost_models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=150, learning_rate=0.5, random_state=RANDOM_STATE),
    "Gradient boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=2,
        min_samples_leaf=8, random_state=RANDOM_STATE
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.08, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    ),
}
rows = []
for name, model in boost_models.items():
    model.fit(Xc_train, yc_train)
    p = model.predict(Xc_test)
    s = model.predict_proba(Xc_test)[:, 1]
    rows.append({"model": name, "precision": precision_score(yc_test, p, zero_division=0),
                 "recall": recall_score(yc_test, p, zero_division=0),
                 "F1": f1_score(yc_test, p, zero_division=0),
                 "ROC-AUC": roc_auc_score(yc_test, s),
                 "average_precision": average_precision_score(yc_test, s)})
pd.DataFrame(rows).set_index("model").round(3)

## 10. Support Vector Machines (SVM)

A linear SVM searches for a decision boundary with a large margin between classes. A kernel can represent nonlinear boundaries without explicitly creating every transformed feature.

- `C` controls the penalty for margin violations: smaller C generally allows a wider margin with more training errors; larger C penalizes errors more strongly.
- `kernel="linear"` gives a linear boundary.
- `kernel="rbf"` can model nonlinear boundaries.
- `gamma` controls the influence range of points for an RBF kernel.
- Scaling is essential for most SVM workflows.
- SVM decision scores are not automatically calibrated probabilities. Probability estimation adds complexity and should be validated if probabilities are required.

In [ ]:
svm_models = {
    "Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", C=1.0, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}
rows = []
for name, model in svm_models.items():
    model.fit(Xc_train, yc_train)
    p = model.predict(Xc_test)
    decision = model.decision_function(Xc_test)
    rows.append({"model": name, "precision": precision_score(yc_test, p, zero_division=0),
                 "recall": recall_score(yc_test, p, zero_division=0),
                 "F1": f1_score(yc_test, p, zero_division=0),
                 "ROC-AUC": roc_auc_score(yc_test, decision),
                 "average_precision": average_precision_score(yc_test, decision)})
pd.DataFrame(rows).set_index("model").round(3)

## 11. Naive Bayes

Naive Bayes applies Bayes' theorem while assuming features are conditionally independent given the class. That assumption is often unrealistic, yet the method can work well as a fast baseline.

Variants:
- **GaussianNB:** assumes each feature follows a Gaussian distribution within each class.
- **MultinomialNB:** commonly used with nonnegative count-like features such as word counts.
- **BernoulliNB:** often used for binary features.

For text classification, Multinomial Naive Bayes is a useful baseline with count or TF-IDF-like nonnegative features. Do not pass arbitrary negative-valued standardized features to MultinomialNB.

In [ ]:
nb = GaussianNB()
nb.fit(Xc_train, yc_train)
nb_pred = nb.predict(Xc_test)
nb_proba = nb.predict_proba(Xc_test)[:, 1]
print("GaussianNB precision:", round(precision_score(yc_test, nb_pred, zero_division=0), 3))
print("GaussianNB recall:", round(recall_score(yc_test, nb_pred, zero_division=0), 3))
print("GaussianNB F1:", round(f1_score(yc_test, nb_pred, zero_division=0), 3))
print("GaussianNB ROC-AUC:", round(roc_auc_score(yc_test, nb_proba), 3))

## 12. Compare algorithms fairly with cross-validation

A single train/test result can be noisy. Cross-validation estimates how performance varies across training/validation folds.

Important fairness rules:
- use the same split folds for candidate models;
- fit preprocessing within each fold;
- define a primary metric before comparing;
- do not use the final test set to select a model;
- report variation, not just the highest score;
- account for training time, prediction latency, interpretability, and maintenance.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate_models = {
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "KNN (k=15)": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=15, weights="distance")),
    ]),
    "Decision tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, class_weight="balanced", random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=180, min_samples_leaf=4, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=120, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE),
    "RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", C=2.0, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "GaussianNB": GaussianNB(),
}
comparison_rows = []
for name, estimator in candidate_models.items():
    scores = cross_validate(
        estimator, Xc_train, yc_train, cv=cv,
        scoring={"ap": "average_precision", "roc_auc": "roc_auc", "f1": "f1", "recall": "recall"},
        n_jobs=-1
    )
    comparison_rows.append({
        "model": name,
        "AP mean": scores["test_ap"].mean(),
        "AP SD": scores["test_ap"].std(),
        "ROC-AUC mean": scores["test_roc_auc"].mean(),
        "F1 mean": scores["test_f1"].mean(),
        "Recall mean": scores["test_recall"].mean(),
    })
comparison = pd.DataFrame(comparison_rows).sort_values("AP mean", ascending=False)
comparison.round(3)

## 13. Hyperparameter tuning with GridSearchCV

A grid search evaluates specified combinations with cross-validation. Tune a small, meaningful search space first, then expand if the evidence suggests it is worthwhile.

Always include a pipeline when the algorithm needs preprocessing. The grid below uses the `model__` prefix because the model is a named pipeline step.

In [ ]:
tune_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
])
param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__solver": ["lbfgs"],
}
search = GridSearchCV(
    tune_pipeline, param_grid, scoring="average_precision",
    cv=cv, n_jobs=-1, refit=True
)
search.fit(Xc_train, yc_train)
print("Best parameters:", search.best_params_)
print("Best mean CV average precision:", round(search.best_score_, 3))

## 14. Feature importance: what can we learn?

Different models expose different forms of interpretation:
- Linear/logistic coefficients describe fitted relationships on the model's feature scale.
- Tree impurity importance can favor features with many possible split points.
- Permutation importance measures score change after shuffling a feature on evaluation data.
- Feature importance is not causal importance. Correlated predictors can share or mask importance.

Use held-out data for importance estimates, check stability across folds, and avoid over-interpreting small differences.

In [ ]:
from sklearn.inspection import permutation_importance

final_candidate = search.best_estimator_
final_candidate.fit(Xc_train, yc_train)
perm = permutation_importance(
    final_candidate, Xc_test, yc_test,
    scoring="average_precision", n_repeats=8,
    random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": Xc.columns,
    "importance_mean": perm.importances_mean,
    "importance_sd": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
importance.head(10)

## 15. Practical model selection guide

Use this as a starting point, not a rigid rule.

- **Need a simple, explainable baseline:** linear or logistic regression.
- **Numeric target with mostly linear relationships:** linear regression; try Ridge/Lasso if coefficients are unstable or many features exist.
- **Small/medium dataset where distance is meaningful:** KNN, after scaling and testing k.
- **Readable nonlinear rules:** a shallow decision tree.
- **Strong general-purpose tabular baseline:** random forest or extra trees.
- **Often strong on structured tabular data:** gradient boosting; tune carefully.
- **Medium-sized data with clear margins:** SVM, especially with scaling.
- **Text/count features or fast probabilistic baseline:** naive Bayes.
- **Need reliable probabilities:** evaluate calibration regardless of algorithm.
- **Need future forecasting:** use time-aware validation and algorithms/features appropriate for temporal structure.

Also consider data size, missingness, categorical handling, training time, inference latency, fairness, interpretability, and maintenance.

## 16. Common mistakes to avoid

1. Comparing models on different splits.
2. Selecting the model based on test-set results.
3. Scaling before splitting or outside cross-validation pipelines.
4. Choosing accuracy for a highly imbalanced target without checking class-specific errors.
5. Assuming tree models never overfit.
6. Assuming a high ROC-AUC guarantees high precision.
7. Interpreting coefficients or feature importance as causal effects.
8. Ignoring prediction latency and operational capacity.
9. Tuning too many hyperparameters on a small dataset.
10. Failing to document target definition, split strategy, metrics, and limitations.

## 17. Practice questions

Try to answer before checking the suggested answers.

1. Why does KNN usually need feature scaling?
2. What is the main difference between bagging and boosting?
3. What does a smaller `C` mean in scikit-learn logistic regression?
4. Why can a deep decision tree overfit?
5. Why might random forest be a good first nonlinear baseline for tabular data?
6. What assumption makes naive Bayes “naive”?
7. Why should a model-selection metric be chosen before tuning?
8. When would you choose MAE over RMSE?
9. Why might a model with the best cross-validation score still not be the best production choice?
10. What should you do if the model ranks cases well but its predicted probabilities are unreliable?

### Suggested answers

1. Distance calculations are dominated by features with larger numeric scales unless features are scaled.
2. Bagging trains learners on resampled data and aggregates them, mainly reducing variance; boosting adds learners sequentially to improve the current ensemble.
3. Smaller `C` means stronger regularization.
4. It can learn noise and overly specific splits that do not generalize.
5. It captures nonlinearities and interactions, usually needs little preprocessing, and averages diverse trees.
6. It assumes features are conditionally independent given the class.
7. Otherwise, model selection can chase a metric that does not match the actual decision and can encourage overfitting to the validation process.
8. When absolute errors in original units are the most relevant and very large errors should not dominate as strongly as they do under squared loss.
9. Latency, interpretability, stability, calibration, maintenance, and error costs may outweigh a small score difference.
10. Evaluate calibration and consider a calibration method using development data, then assess the final process on untouched evaluation data.

## 18. Capstone — select a supervised model

### Business scenario
A team must rank cases for manual review. The positive class is rare, and review capacity is limited.

### Deliverables
1. Write a one-paragraph problem statement.
2. State the target, prediction time, and unit of analysis.
3. Explain the split strategy and leakage risks.
4. Establish a dummy baseline.
5. Compare logistic regression, KNN, a decision tree, random forest, gradient boosting, SVM, and naive Bayes using cross-validation.
6. Choose a primary metric and justify it.
7. Select a candidate based on CV results and operational constraints.
8. Tune the selected model without touching the final test set.
9. Evaluate once on the test set.
10. Discuss precision, recall, false-positive workload, calibration, feature importance, limitations, and monitoring.

### Model card outline
- Model name/version
- Intended use and prohibited use
- Target and prediction time
- Data sources and split method
- Baseline and selected model
- Primary metric and test results
- Known limitations and subgroup evaluation
- Human review / fallback
- Monitoring and retraining plan

## 19. Readiness checklist

- [ ] Explain the intuition and trade-offs of each algorithm.
- [ ] Fit and diagnose linear regression.
- [ ] Explain regularization in Ridge and Lasso.
- [ ] Interpret logistic-regression probabilities and coefficients cautiously.
- [ ] Explain KNN distance and the role of k.
- [ ] Tune decision-tree complexity.
- [ ] Distinguish bagging, random forests, and boosting.
- [ ] Explain the role of C and kernels in SVMs.
- [ ] Explain the conditional-independence assumption in naive Bayes.
- [ ] Compare models with the same cross-validation folds and metric.
- [ ] Tune hyperparameters without using the final test set.
- [ ] Explain why model selection depends on operational requirements as well as metrics.

### Next module
**Module 7 — Unsupervised Learning and Dimensionality Reduction:** K-means, hierarchical clustering, DBSCAN, PCA, feature scaling, cluster evaluation, anomaly detection, and practical applications.